In [11]:
import pandas as pd
import geopandas as gpd

boroughs = gpd.read_file("../data/nyu-2451-41649-geojson.json")[["boroname", "geometry"]]
boroughs = boroughs.rename(columns={"boroname": "boro_name"}).to_crs("EPSG:4326")

df = pd.read_parquet("../data/cleaned.parquet")

mask = df["borough"].isna() & df["latitude"].notna() & df["longitude"].notna()

pts = gpd.GeoDataFrame(
    df[mask],
    geometry=gpd.points_from_xy(df.loc[mask, "longitude"], df.loc[mask, "latitude"]),
    crs="EPSG:4326",
)

joined = gpd.sjoin(pts, boroughs, how="left", predicate="within")
joined = joined[~joined.index.duplicated()]

df.loc[joined.index, "borough"] = joined["boro_name"].str.upper()

In [12]:
df.head()

,crash date,crash time,borough,zip code,latitude,longitude,location,on street name,cross street name,off street name,...,contributing factor vehicle 5,collision_id,vehicle type code 1,vehicle type code 2,vehicle type code 3,vehicle type code 4,vehicle type code 5,month,hour_of_collision,time_period
0,2020-08-29,1900-01-01 15:40:00,bronx,10466.0,40.89210,-73.833760,point (-73.83376 40.8921),pratt avenue,strang avenue,NaN,...,NaN,4342908,sedan,station wagon/sport utility vehicle,NaN,NaN,NaN,august,15,afternoon
1,2020-08-29,1900-01-01 21:00:00,brooklyn,11221.0,40.69050,-73.919914,point (-73.919914 40.6905),bushwick avenue,palmetto street,NaN,...,NaN,4343555,sedan,sedan,NaN,NaN,NaN,august,21,evening
2,2020-08-29,1900-01-01 18:20:00,MANHATTAN,NaN,40.81650,-73.946556,point (-73.946556 40.8165),8 avenue,NaN,NaN,...,NaN,4343142,station wagon/sport utility vehicle,NaN,NaN,NaN,NaN,august,18,evening
3,2020-08-29,1900-01-01 00:00:00,bronx,10459.0,40.82472,-73.892960,point (-73.89296 40.82472),NaN,NaN,1047 simpson street,...,NaN,4343588,station wagon/sport utility vehicle,station wagon/sport utility vehicle,sedan,motorcycle,NaN,august,0,night
4,2020-08-29,1900-01-01 17:10:00,brooklyn,11203.0,40.64989,-73.933890,point (-73.93389 40.64989),NaN,NaN,4609 snyder avenue,...,NaN,4342953,sedan,sedan,NaN,NaN,NaN,august,17,afternoon


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74881 entries, 0 to 74880
Data columns (total 32 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   crash date                     74881 non-null  datetime64[us]
 1   crash time                     74881 non-null  datetime64[us]
 2   borough                        70316 non-null  str           
 3   zip code                       49134 non-null  float64       
 4   latitude                       68935 non-null  float64       
 5   longitude                      68935 non-null  float64       
 6   location                       68935 non-null  str           
 7   on street name                 55444 non-null  str           
 8   cross street name              35681 non-null  str           
 9   off street name                19437 non-null  str           
 10  number of persons injured      74881 non-null  int64         
 11  number of persons killed  

In [14]:
df["borough"] = df["borough"].str.lower()

In [15]:
df.to_parquet("../data/clean2.parquet")

In [16]:
df.head()

,crash date,crash time,borough,zip code,latitude,longitude,location,on street name,cross street name,off street name,...,contributing factor vehicle 5,collision_id,vehicle type code 1,vehicle type code 2,vehicle type code 3,vehicle type code 4,vehicle type code 5,month,hour_of_collision,time_period
0,2020-08-29,1900-01-01 15:40:00,bronx,10466.0,40.89210,-73.833760,point (-73.83376 40.8921),pratt avenue,strang avenue,NaN,...,NaN,4342908,sedan,station wagon/sport utility vehicle,NaN,NaN,NaN,august,15,afternoon
1,2020-08-29,1900-01-01 21:00:00,brooklyn,11221.0,40.69050,-73.919914,point (-73.919914 40.6905),bushwick avenue,palmetto street,NaN,...,NaN,4343555,sedan,sedan,NaN,NaN,NaN,august,21,evening
2,2020-08-29,1900-01-01 18:20:00,manhattan,NaN,40.81650,-73.946556,point (-73.946556 40.8165),8 avenue,NaN,NaN,...,NaN,4343142,station wagon/sport utility vehicle,NaN,NaN,NaN,NaN,august,18,evening
3,2020-08-29,1900-01-01 00:00:00,bronx,10459.0,40.82472,-73.892960,point (-73.89296 40.82472),NaN,NaN,1047 simpson street,...,NaN,4343588,station wagon/sport utility vehicle,station wagon/sport utility vehicle,sedan,motorcycle,NaN,august,0,night
4,2020-08-29,1900-01-01 17:10:00,brooklyn,11203.0,40.64989,-73.933890,point (-73.93389 40.64989),NaN,NaN,4609 snyder avenue,...,NaN,4342953,sedan,sedan,NaN,NaN,NaN,august,17,afternoon


In [19]:
df = df.drop_duplicates("collision_id")

# non-null isn't the same as valid: check for zeros and out-of-NYC points
has_xy = df["latitude"].notna() & df["longitude"].notna()
valid = (has_xy
         & df["latitude"].between(40.49, 40.92)
         & df["longitude"].between(-74.27, -73.69))
print(has_xy.sum(), valid.sum(), len(df) - valid.sum())

df["severity"] = (1 + 3*df["number of persons injured"]
                    + 10*df["number of persons killed"])

import geopandas as gpd, numpy as np
from sklearn.cluster import DBSCAN

g = gpd.GeoDataFrame(
    df[valid],
    geometry=gpd.points_from_xy(df.loc[valid, "longitude"], df.loc[valid, "latitude"]),
    crs=4326).to_crs(2263)            # NY State Plane, units = feet

xy = np.c_[g.geometry.x, g.geometry.y]
g["cluster"] = DBSCAN(eps=300, min_samples=15).fit_predict(xy)   # ~90 m radius; -1 = noise

def top(s): return s.dropna().mode().iloc[0] if s.notna().any() else None

hot = (g[g.cluster >= 0].groupby("cluster")
         .agg(crashes=("collision_id", "count"),
              severity=("severity", "sum"),
              killed=("number of persons killed", "sum"),
              on_street=("on street name", top),
              cross_street=("cross street name", top)).reset_index()
         .sort_values("severity", ascending=False))

68935 68872 6009


In [25]:
hot["crashes"].sum()

np.int64(20229)

In [22]:
hot.info()

<class 'pandas.DataFrame'>
Index: 695 entries, 161 to 663
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   cluster       695 non-null    int64
 1   crashes       695 non-null    int64
 2   severity      695 non-null    int64
 3   killed        695 non-null    int64
 4   on_street     692 non-null    str  
 5   cross_street  655 non-null    str  
dtypes: int64(4), str(2)
memory usage: 56.9 KB
